# Databricks notebook source
 %md
 # Landing (Parquet) -> Bronze (Delta)

 One notebook, all domains. ADF: Lookup -> ForEach -> this notebook.
 Assumes the catalog, schema, and tables already exist
 (`01_Bronze_Catalog_Schema_Creation.sql`, `02_bronze_external_tables.sql`).

 Loads every landing date that isn't in Bronze yet, so a missed day
 self-heals on the next run.


In [0]:
dbutils.widgets.text("domain",           "CM")
dbutils.widgets.text("storageAccName",   "stclintraildev001")
dbutils.widgets.text("srcContainerName", "landing")
dbutils.widgets.text("tgtCatalog",       "clintrail_dev")
dbutils.widgets.text("tgtSchema",        "clintrail_bronze")
dbutils.widgets.text("tgtTableName",     "cm")

domain    = dbutils.widgets.get("domain")
account   = dbutils.widgets.get("storageAccName")
container = dbutils.widgets.get("srcContainerName")

In [0]:
DATE_COL = "bronze_ingest_date"
ROOT     = f"abfss://{container}@{account}.dfs.core.windows.net/src/{domain}"
TARGET   = (f"{dbutils.widgets.get('tgtCatalog')}"
            f".{dbutils.widgets.get('tgtSchema')}"
            f".{dbutils.widgets.get('tgtTableName')}")

print(ROOT)
print(TARGET)


What to load = the dates in landing, MINUS the dates already in Bronze.
Both are observed facts (one from the filesystem, one from the table), so there
is no remembered state to drift. Miss a day and the next run picks it up.

{"2026-07-08", "2026-07-09", "2026-07-10"}  <- folder names ingest_date=YYYY-MM-DD

{"2026-07-08", "2026-07-09"} str() matters: r[0] is a datetime.date, landing holds strings, and a date never equals a str. Without it the sets share nothing and Bronze reloads all history.


In [0]:
landing = {f.name.split("=", 1)[1].rstrip("/")
           for f in dbutils.fs.ls(ROOT) if f.name.startswith("ingest_date=")}
loaded = {str(r[0]) for r in spark.table(TARGET).select(DATE_COL).distinct().collect()
          if r[0] is not None}
todo = sorted(landing.difference(loaded))
print(landing)
print(loaded)
print(todo)

In [0]:
print(f"{domain}: landing={len(landing)}  bronze={len(loaded)}  to load={todo}")
if not todo:
    dbutils.notebook.exit(f"OK|{domain}|0 rows")

One full path per date we need to load.
["abfss://.../src/DM/ingest_date=2026-07-10"]

In [0]:
paths = [f"{ROOT}/ingest_date={d}" for d in todo]
print(paths)

In [0]:
df = (spark.read
        .option("basePath", ROOT)
        .parquet(*paths)                      # * = pass each path as its own argument
        .withColumnRenamed("ingest_date", DATE_COL))

n = df.count()
# df.display()

In [0]:
date_list = ", ".join(f"'{d}'" for d in todo)
# print(date_list)
predicate = f"{DATE_COL} in ({date_list})"
# print(predicate)
print("replaceWhere:", predicate)

Delta deletes the rows matching the predicate and writes the new ones, in ONE
transaction. So re-running a date replaces it instead of duplicating it, and every
other date is untouched. mergeSchema lets Bronze absorb source schema drift.

In [0]:
(df.write.format("delta")
   .mode("overwrite")
   .option("mergeSchema", "true")
   .option("replaceWhere", predicate)
   .saveAsTable(TARGET))

print(f"wrote {n} rows to {TARGET}")